In [1]:
# ============================================================
# Part 1. Common imports + utilities + forcing + Weibull
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import curve_fit
from scipy.signal import find_peaks, savgol_filter
from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import UnivariateSpline


# ============================================================
# 1) Common default CONFIG
# ============================================================

COMMON_CONFIG = {
    # --------------------------
    # Basic columns
    # --------------------------
    "sheet_name": 0,
    "col_year": "year",
    "col_doy": "doy",
    "col_plot": "plot",
    "col_gpp_candidates": [
        "gpp", "GPP", "gpp_obs", "gpp_sim", "GPP_obs", "GPP_sim",
        "gcc", "GCC", "smooth_gcc", "Smooth_GCC"
    ],

    # --------------------------
    # Plot metadata
    # --------------------------
    "plots": ["P06", "P20", "P13", "P08", "P17", "P19", "P11", "P04", "P16", "P10"],
    "co2_by_plot": [0, 0, 0, 0, 0, 500, 500, 500, 500, 500],
    "warming_by_plot": [0, 2.25, 4.5, 6.75, 9, 0, 2.25, 4.5, 6.75, 9],

    # --------------------------
    # Forcing
    # --------------------------
    "path_forcing": "../../data_results/1_data_source/1_in_alltreat/",
    "forcing_template": "{plot}/SPRUCE_forcing.txt",
    "gs_doy_min": 150,
    "gs_doy_max": 300,

    # --------------------------
    # Curve methods
    # --------------------------
    "curve_keys": ["weibull", "hants", "savgol", "spline", "final_hss", "final_whs"],

    # --------------------------
    # AMP-threshold phenology
    # --------------------------
    "dormant_q": 0.05,
    "summer_window": (180, 260),
    "amp_fracs": [0.10, 0.20, 0.30, 0.25],
    "use_longest_segment": True,
    "min_consecutive_days": 5,

    # --------------------------
    # Temperature gating for CUP
    # --------------------------
    "tair_threshold_options_start": [5.0],
    "tair_threshold_options_end": [0.0],
    "run_all_tair_threshold_pairs": False,
    "plot_tair_threshold_start": 5.0,
    "plot_tair_threshold_end": 0.0,
    "tair_gate_mode": "gate_crossings",
    "require_crossings_in_warm": True,

    # --------------------------
    # Threshold scope
    # --------------------------
    "threshold_scope": "annual",
    "threshold_year_pool": "pretreatment",

    # --------------------------
    # Dormant minimum definition
    # --------------------------
    "gppmin_mode": "quantile",
    "tair_for_gppmin": 0.0,
    "gppmin_fallback_quantile": 0.05,

    # --------------------------
    # Smoothers
    # --------------------------
    "hants": {
        "K": 2,
        "robust_iter": 6,
        "zcut": 3.0,
        "ridge": 1e-6,
        "clip_nonnegative": True,
    },
    "savgol": {
        "window": 21,
        "polyorder": 3,
        "clip_nonnegative": True,
    },
    "spline": {
        "s": None,
        "robust_iter": 3,
        "zcut": 3.0,
        "clip_nonnegative": True,
    },

    # --------------------------
    # Weibull
    # --------------------------
    "denoise": {
        "enable": True,
        "method": "gaussian_then_median",
        "gaussian_sigma": 2.0,
        "median_window": 7,
        "clip_nonnegative": True,
    },
    "weibull_p0_single": [-0.9, 7.8, 567.5, 215.4, 2.7],
    "weibull_p0_seg1":   [-1.3, 6.6, 168.2, -2600150.1, -46311.2],
    "weibull_p0_seg2":   [-1.3, 8.2, 554.04, 217.6, 2.3],

    "weibull_p0_special": {
        ("sphagnum", "P13", 2021, "seg2"): [0.03, 0.43, 430, 96.6, 2.2],
        ("sphagnum", "P06", 2014, "seg1"): [-1.3, 2.9, 131.1, -18974646.5, -243311.8],
        ("sphagnum", "P06", 2014, "seg2"): [0.7, 1.14, 383.5, 57.1, 2.5],
        ("sphagnum", "P17", 2016, "seg1"): [-1.3, 6.6, 168.2, -2600150.1, -46311.2],
        ("sphagnum", "P17", 2018, "seg2"): [0.0512, 0.811, 414.366, 72.39, 2.844],
        ("sphagnum", "P17", 2021, "seg2"): [0.0512, 0.811, 414.366, 72.39, 2.844],
        ("sphagnum", "P04", 2016, "single"): [-1.2, 14.4, 592.1, 221.6, 3.2],
        ("sphagnum", "P04", 2019, "seg2"): [0.0512, 0.811, 414.366, 72.39, 2.844],
        ("sphagnum", "P04", 2021, "seg1"): [-0.9, 7.8, 567.5, 215.4, 2.7],
        ("sphagnum", "P19", 2018, "seg1"): [0.312, 0.96, 93.2, -4746124.23, 136849.36],
        ("sphagnum", "P19", 2017, "seg2"): [0.312, 0.96, 93.2, -4746124.23, 136849.36],
        ("sphagnum", "P16", 2014, "single"): [0.002487, 0.1, 34804.6, 17297.09, -424.2],
        ("sphagnum", "P10", 2018, "seg1"): [0.024, 0.14, 109.11, -4112675.3, 96379.85],
        ("sphagnum", "P10", 2018, "seg2"): [0.0148, 0.138, 426.95, 89.93, 3.104],
        ("shrub", "P10", 2021, "seg2"): [-378.8, 380.9, 626.6, 5376.6, 1.029],
        ("shrub", "P08", 2021, "seg2"): [-314.6, 315.7, 560.2, 8447.5, 1.015],

        # wildcard keys
        ("*", "P13", 2021, "seg2"): [0.03, 0.43, 430, 96.6, 2.2],
        ("sphagnum", "P17", "*", "seg2"): [0.0512, 0.811, 414.366, 72.39, 2.844],
    },

    "weibull_bounds_special": {
        ("sphagnum", "P04", 2021, "seg1"): (
            [-1.0, 0.1, 150.0, 1.0, 1.05],
            [0.0, 1.5, 630.0, 300.0, 10.0],
        ),
        ("sphagnum", "P04", 2021, "seg2"): (
            [0.01, -0.01, 150.0, 1.0, 1.05],
            [2.0, 10.0, 530.0, 100.0, 10.0],
        ),
    },
    "weibull_maxfev": 300000,

    # --------------------------
    # Pretreatment anomaly
    # --------------------------
    "pretreat_enable": True,
    "pretreat_mode": "years_mean",
    "pretreat_years": [2011, 2012, 2013],
    "pretreat_grouping": "method_plot",

    # --------------------------
    # Output and plot
    # --------------------------
    "dpi": 250,
    "fig_cell_w": 2.2,
    "fig_cell_h": 1.8,
    "raw_point_size": 4,
    "raw_alpha": 0.25,
    "plot_cup_lines": True,
    "plot_tair0_vlines": True,

    # whether to calculate alpha
    "calc_alpha": True,
    "y_label": "GPP",
}


# ============================================================
# 2) General utilities
# ============================================================

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)


def is_leap(y: int) -> bool:
    return (y % 4 == 0 and y % 100 != 0) or (y % 400 == 0)


def mad_z(resid):
    resid = np.asarray(resid, float)
    med = np.nanmedian(resid)
    mad = np.nanmedian(np.abs(resid - med)) + 1e-12
    return 0.6745 * (resid - med) / mad


def to_full_daily_series(doy, value, year):
    doy = np.asarray(doy, int)
    value = np.asarray(value, float)

    n = 366 if is_leap(int(year)) else 365
    doy_full = np.arange(1, n + 1, dtype=int)

    s = pd.Series(value, index=doy)
    s = s[~s.index.duplicated(keep="first")]
    s_full = s.reindex(doy_full)

    orig_mask = np.isfinite(s_full.to_numpy())
    y_full = pd.Series(s_full.to_numpy()).interpolate(limit_direction="both").to_numpy()
    y_full = np.clip(y_full, 0, None)

    return doy_full, y_full, orig_mask


def detect_and_normalize(df_raw, cfg):
    df = df_raw.copy()
    df.columns = [c.strip() if isinstance(c, str) else c for c in df.columns]

    ycol = cfg["col_year"]
    dcol = cfg["col_doy"]
    pcol = cfg["col_plot"]

    if ycol not in df.columns or dcol not in df.columns:
        raise ValueError(f"Input must contain columns '{ycol}' and '{dcol}'.")

    # LONG format
    if pcol in df.columns:
        value_col = None
        for c in cfg["col_gpp_candidates"]:
            if c in df.columns:
                value_col = c
                break

        if value_col is not None:
            out = df[[ycol, dcol, pcol, value_col]].copy()
            out = out.rename(columns={
                ycol: "year",
                dcol: "doy",
                pcol: "plot",
                value_col: "value",
            })
            out["year"] = out["year"].astype(int)
            out["doy"] = out["doy"].astype(int)
            out["plot"] = out["plot"].astype(str)
            out["value"] = pd.to_numeric(out["value"], errors="coerce").fillna(0.0)
            out["value"] = np.clip(out["value"].to_numpy(), 0, None)
            return out

    # WIDE format
    plot_cols = [p for p in cfg["plots"] if p in df.columns]
    if len(plot_cols) == 0:
        raise ValueError(
            "Cannot detect input format.\n"
            f"LONG needs columns: {ycol}, {dcol}, {pcol}, and value column.\n"
            f"WIDE needs plot columns: {cfg['plots']}."
        )

    out = df[[ycol, dcol] + plot_cols].copy()
    out = out.rename(columns={ycol: "year", dcol: "doy"})
    out["year"] = out["year"].astype(int)
    out["doy"] = out["doy"].astype(int)

    out = out.melt(id_vars=["year", "doy"], var_name="plot", value_name="value")
    out["plot"] = out["plot"].astype(str)
    out["value"] = pd.to_numeric(out["value"], errors="coerce").fillna(0.0)
    out["value"] = np.clip(out["value"].to_numpy(), 0, None)

    return out


def _nanmean_safe(vals):
    vals = [v for v in vals if np.isfinite(v)]
    return float(np.mean(vals)) if len(vals) > 0 else np.nan


def ensemble_cup_from_metrics(row, members, suffix):
    suf = str(suffix)
    cs = _nanmean_safe([row.get(f"cup_start_{m}_{suf}", np.nan) for m in members])
    ce = _nanmean_safe([row.get(f"cup_end_{m}_{suf}", np.nan) for m in members])
    cup = float(ce - cs + 1) if np.isfinite(cs) and np.isfinite(ce) and ce >= cs else np.nan
    return cs, ce, cup


def ensemble_gppmax_from_metrics(row, members):
    gmax = _nanmean_safe([row.get(f"gpp_max_{m}", np.nan) for m in members])
    doy = _nanmean_safe([row.get(f"doy_gppmax_{m}", np.nan) for m in members])
    return gmax, doy


def alpha_from(value_sum, gpp_max, cup, calc_alpha=True):
    if not calc_alpha:
        return np.nan
    return (
        value_sum / (gpp_max * cup)
        if np.isfinite(value_sum)
        and np.isfinite(gpp_max)
        and np.isfinite(cup)
        and gpp_max > 0
        and cup > 0
        else np.nan
    )


# ============================================================
# 3) Forcing functions
# ============================================================

def load_forcing_daily(plot_id, cfg):
    fpath = os.path.join(cfg["path_forcing"], cfg["forcing_template"].format(plot=plot_id))

    if not os.path.exists(fpath):
        raise FileNotFoundError(f"Cannot find forcing file for plot {plot_id}: {fpath}")

    df = pd.read_csv(fpath, sep="\t")

    for need in ["year", "doy", "hour", "Tair"]:
        if need not in df.columns:
            raise ValueError(f"Forcing file missing column '{need}': {fpath}")

    df["time"] = (
        pd.to_datetime(df["year"].astype(str) + df["doy"].astype(str), format="%Y%j")
        + pd.to_timedelta(df["hour"], unit="h")
    )

    df = df.set_index("time")
    daily = df.resample("D").mean(numeric_only=True).dropna(subset=["Tair"])
    daily = daily.reset_index(drop=False)

    daily["year"] = daily["time"].dt.year.astype(int)
    daily["doy"] = daily["time"].dt.dayofyear.astype(int)

    return daily[["year", "doy", "Tair"]]


def summarize_temp(daily_forcing, year, gs_doy_min, gs_doy_max):
    sub = daily_forcing[daily_forcing["year"] == int(year)].copy()

    if sub.empty:
        return np.nan, np.nan, np.nan

    temp_mean = float(sub["Tair"].mean())
    temp_max = float(sub["Tair"].max())

    gs = sub[(sub["doy"] >= int(gs_doy_min)) & (sub["doy"] <= int(gs_doy_max))]
    temp_gs_mean = float(gs["Tair"].mean()) if not gs.empty else np.nan

    return temp_mean, temp_gs_mean, temp_max


def tair_full_for_year(daily_forcing, year, doy_full):
    if daily_forcing is None:
        return np.full(len(doy_full), np.nan, dtype=float)

    sub = daily_forcing[daily_forcing["year"] == int(year)].copy()

    if sub.empty:
        return np.full(len(doy_full), np.nan, dtype=float)

    m = dict(zip(sub["doy"].astype(int).to_numpy(), sub["Tair"].to_numpy()))
    return np.array([m.get(int(d), np.nan) for d in doy_full], dtype=float)


def tair_mask_for_year(daily_forcing, year, doy_full, tair_thr):
    tair = tair_full_for_year(daily_forcing, year, doy_full)
    return np.isfinite(tair) & (tair >= float(tair_thr))


def tair0_vline_doys(tair_full, doy_full):
    if tair_full is None:
        return np.nan, np.nan

    t = np.asarray(tair_full, float)
    d = np.asarray(doy_full, int)

    ok = np.isfinite(t)
    if ok.sum() == 0:
        return np.nan, np.nan

    ge0 = ok & (t >= 0.0)

    if ge0.sum() == 0:
        return np.nan, np.nan

    spring0 = float(d[np.argmax(ge0)])
    fall0 = float(d[np.where(ge0)[0][-1]])

    return spring0, fall0


# ============================================================
# 4) Weibull fitting
# ============================================================

def rolling_median(x, win=7):
    x = np.asarray(x, dtype=float)

    if win is None or int(win) <= 1:
        return x.copy()

    win = int(win)
    if win % 2 == 0:
        win += 1

    s = pd.Series(x)
    y = s.rolling(win, center=True, min_periods=max(1, win // 3)).median()
    y = y.interpolate(limit_direction="both").to_numpy()

    return y


def denoise_series(value, cfg_denoise):
    value = np.asarray(value, dtype=float)
    out = pd.Series(value).interpolate(limit_direction="both").to_numpy()

    if not cfg_denoise.get("enable", True):
        return np.clip(out, 0, None) if cfg_denoise.get("clip_nonnegative", True) else out

    method = cfg_denoise.get("method", "gaussian")

    if method == "none":
        pass
    elif method == "gaussian":
        sig = float(cfg_denoise.get("gaussian_sigma", 2.0))
        if sig > 0:
            out = gaussian_filter1d(out, sigma=sig, mode="nearest")
    elif method == "median":
        win = int(cfg_denoise.get("median_window", 7))
        out = rolling_median(out, win=win)
    elif method == "gaussian_then_median":
        sig = float(cfg_denoise.get("gaussian_sigma", 2.0))
        win = int(cfg_denoise.get("median_window", 7))
        if sig > 0:
            out = gaussian_filter1d(out, sigma=sig, mode="nearest")
        out = rolling_median(out, win=win)
    else:
        raise ValueError(f"Unknown denoise method: {method}")

    if cfg_denoise.get("clip_nonnegative", True):
        out = np.clip(out, 0, None)

    return out


def func_weibull(t, y0, a, x0, b, c):
    t = np.asarray(t, dtype=float)
    threshold = x0 - b * (c - 1) / c
    y = np.zeros_like(t) + y0

    condition = t <= threshold

    if np.any(condition):
        y[condition] = (
            y0
            + a
            * np.power((c - 1) / c, (1 - c) / c)
            * np.power(
                np.abs((t[condition] - x0) / b + np.power((c - 1) / c, 1 / c)),
                c - 1,
            )
            * np.exp(
                -np.power(
                    np.abs((t[condition] - x0) / b + np.power((c - 1) / c, 1 / c)),
                    c,
                )
                + (c - 1) / c
            )
        )

    return y


def seasonal_function(t, c1, c2, c3, c4, c5):
    t = np.asarray(t, dtype=float)
    n = len(t)
    omega = 6 * np.pi / n

    return (
        c1
        + c2 * np.sin(omega * t)
        + c3 * np.cos(omega * t)
        + c4 * np.sin(2 * omega * t)
        + c5 * np.cos(2 * omega * t)
    )


def test_peaks(dat_value):
    dat_value = np.asarray(dat_value, dtype=float)
    n = len(dat_value)

    if n < 30:
        return 1, np.array([], dtype=int), np.array([], dtype=int)

    value3 = np.tile(dat_value, 3)
    t3 = np.arange(1, n * 3 + 1)

    init_guess_test = [0.1, 1000, 1, 0.8, 0.8]

    try:
        params, _ = curve_fit(
            seasonal_function,
            t3,
            value3,
            p0=init_guess_test,
            maxfev=30000,
        )
        fitted3 = seasonal_function(t3, *params)
        fitted1 = fitted3[:n]
    except Exception:
        return 1, np.array([], dtype=int), np.array([], dtype=int)

    peaks_test, _ = find_peaks(fitted1)
    troughs, _ = find_peaks(-fitted1)

    if len(peaks_test) > 1:
        peaks_test = peaks_test[peaks_test > 60]
        troughs = troughs[troughs > 60]

    check_peak = 1

    if len(peaks_test) > 1:
        primary_peak = np.max(fitted1[peaks_test])
        secondary_peak = np.sort(fitted1[peaks_test])[-2]
        sunk_value = np.max(fitted1[troughs]) if len(troughs) else np.min(fitted1)

        ratio_1 = secondary_peak / max(primary_peak, 1e-12)
        ratio_2 = sunk_value / max(secondary_peak, 1e-12)

        if ratio_1 >= 0.25 and ratio_2 <= 0.9:
            check_peak = 2

    return check_peak, peaks_test, troughs


def _lookup_special(d, sps, plot, year, segment):
    keys = [
        (str(sps), str(plot), int(year), str(segment)),
        (str(sps), str(plot), "*", str(segment)),
        ("*", str(plot), int(year), str(segment)),
        ("*", str(plot), "*", str(segment)),
    ]

    for k in keys:
        if k in d:
            return d[k]

    return None


def _get_weibull_p0(cfg, sps, plot, year, segment):
    ov = cfg.get("weibull_p0_special", {})
    val = _lookup_special(ov, sps, plot, year, segment)

    if val is not None:
        return list(val)

    if segment == "single":
        return list(cfg["weibull_p0_single"])
    if segment == "seg1":
        return list(cfg["weibull_p0_seg1"])
    if segment == "seg2":
        return list(cfg["weibull_p0_seg2"])

    raise ValueError("Unknown segment")


def _get_weibull_bounds(cfg, sps, plot, year, segment):
    bv = cfg.get("weibull_bounds_special", {})
    val = _lookup_special(bv, sps, plot, year, segment)

    if val is not None:
        lo, hi = val
        return np.asarray(lo, float), np.asarray(hi, float)

    return None


def fit_weibull(doy, value_for_fit, p0, bounds=None, maxfev=300000):
    doy = np.asarray(doy, dtype=float)
    y = np.asarray(value_for_fit, dtype=float)

    yy = np.clip(y, 0, None)
    sigma = 1.0 + (yy / (np.nanpercentile(yy, 80) + 1e-6)) ** 2

    if bounds is None:
        params, _ = curve_fit(
            func_weibull,
            doy,
            y,
            p0=p0,
            method="lm",
            maxfev=maxfev,
            sigma=sigma,
            absolute_sigma=False,
        )
    else:
        lo, hi = bounds
        p0 = np.asarray(p0, float)
        p0 = np.clip(p0, lo + 1e-12, hi - 1e-12)

        params, _ = curve_fit(
            func_weibull,
            doy,
            y,
            p0=p0,
            bounds=(lo, hi),
            method="trf",
            maxfev=maxfev,
            sigma=sigma,
            absolute_sigma=False,
        )

    fitted = func_weibull(doy, *params)
    fitted = np.clip(fitted, 0, None)

    return fitted


def fit_year_weibull_single_or_double(doy_full, y_full, cfg, sps, plot, year):
    doy = np.asarray(doy_full, float)
    y_raw = np.clip(np.asarray(y_full, float), 0, None)

    y_fit = denoise_series(y_raw, cfg["denoise"])

    check_peak, _, troughs = test_peaks(y_fit)
    split_idx = int(troughs[0]) if check_peak > 1 and len(troughs) > 0 else None

    maxfev = int(cfg.get("weibull_maxfev", 300000))

    p0_single = _get_weibull_p0(cfg, sps, plot, year, "single")
    p0_1 = _get_weibull_p0(cfg, sps, plot, year, "seg1")
    p0_2 = _get_weibull_p0(cfg, sps, plot, year, "seg2")

    bnd_single = _get_weibull_bounds(cfg, sps, plot, year, "single")
    bnd_1 = _get_weibull_bounds(cfg, sps, plot, year, "seg1")
    bnd_2 = _get_weibull_bounds(cfg, sps, plot, year, "seg2")

    if split_idx is not None and 10 < split_idx < len(y_fit) - 10:
        try:
            fit1 = fit_weibull(
                doy[:split_idx],
                y_fit[:split_idx],
                p0_1,
                bounds=bnd_1,
                maxfev=maxfev,
            )
            fit2 = fit_weibull(
                doy[split_idx:],
                y_fit[split_idx:],
                p0_2,
                bounds=bnd_2,
                maxfev=maxfev,
            )
            fitted = np.concatenate([fit1, fit2])
            return fitted, True, split_idx, y_fit
        except Exception:
            pass

    try:
        fitted = fit_weibull(
            doy,
            y_fit,
            p0_single,
            bounds=bnd_single,
            maxfev=maxfev,
        )
        return fitted, False, None, y_fit
    except Exception:
        return y_raw.copy(), False, None, y_fit

In [2]:
# ============================================================
# Part 2. Common smoothers + phenology + output + plotting + main
# ============================================================

# ============================================================
# 5) Smoothers
# ============================================================

def smooth_hants(doy_full, y_full, cfg):
    K = int(cfg.get("K", 2))
    iters = int(cfg.get("robust_iter", 6))
    zcut = float(cfg.get("zcut", 3.0))
    ridge = float(cfg.get("ridge", 1e-6))

    t = np.asarray(doy_full, float)
    y = np.asarray(y_full, float)
    n = len(t)
    P = float(n)

    cols = [np.ones(n)]
    for k in range(1, K + 1):
        w = 2.0 * np.pi * k / P
        cols.append(np.sin(w * t))
        cols.append(np.cos(w * t))

    X = np.column_stack(cols)
    mask = np.isfinite(y)

    if mask.sum() < (2 * K + 2):
        return None

    for _ in range(iters):
        xm = X[mask]
        ym = y[mask]

        A = xm.T @ xm + ridge * np.eye(X.shape[1])
        b = xm.T @ ym

        try:
            beta = np.linalg.solve(A, b)
        except Exception:
            return None

        yhat = X @ beta
        resid = y - yhat
        z = mad_z(resid[mask])
        keep = np.abs(z) <= zcut

        new_mask = mask.copy()
        idx = np.where(mask)[0]
        new_mask[idx] = keep

        if np.array_equal(new_mask, mask):
            break

        mask = new_mask

        if mask.sum() < (2 * K + 2):
            break

    if mask.sum() < (2 * K + 2):
        return None

    xm = X[mask]
    ym = y[mask]

    A = xm.T @ xm + ridge * np.eye(X.shape[1])
    b = xm.T @ ym

    try:
        beta = np.linalg.solve(A, b)
    except Exception:
        return None

    yhat = X @ beta

    if cfg.get("clip_nonnegative", True):
        yhat = np.clip(yhat, 0, None)

    return yhat


def smooth_savgol(y_full, cfg):
    y = np.asarray(y_full, float)
    n = len(y)

    win = int(cfg.get("window", 21))
    poly = int(cfg.get("polyorder", 3))

    if win % 2 == 0:
        win += 1

    win = min(win, n if n % 2 == 1 else n - 1)
    win = max(win, poly + 2 + (poly + 2) % 2)

    if win >= n or win < (poly + 2):
        yhat = pd.Series(y).rolling(7, center=True, min_periods=3).median()
        yhat = yhat.interpolate(limit_direction="both").to_numpy()
    else:
        try:
            yhat = savgol_filter(y, window_length=win, polyorder=poly, mode="interp")
        except Exception:
            return None

    if cfg.get("clip_nonnegative", True):
        yhat = np.clip(yhat, 0, None)

    return yhat


def robust_spline_fit(x, y, s=None, n_iter=3, zcut=3.0):
    x = np.asarray(x, float)
    y = np.asarray(y, float)

    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]

    if len(x) < 10:
        return None

    idx = np.argsort(x)
    x = x[idx]
    y = y[idx]

    if s is None:
        var = float(np.nanvar(y)) if np.isfinite(np.nanvar(y)) else 1.0
        s = 0.5 * len(x) * var

    kept = np.ones(len(x), dtype=bool)

    for _ in range(int(n_iter)):
        xs = x[kept]
        ys = y[kept]

        if len(xs) < 10:
            break

        spl = UnivariateSpline(xs, ys, s=s, k=3)
        yhat = spl(x)

        resid = y - yhat
        z = mad_z(resid[kept])
        keep = np.abs(z) <= float(zcut)

        idxk = np.where(kept)[0]
        new_kept = kept.copy()
        new_kept[idxk] = keep

        if np.array_equal(new_kept, kept):
            break

        kept = new_kept

    xs = x[kept]
    ys = y[kept]

    if len(xs) < 10:
        return None

    spl = UnivariateSpline(xs, ys, s=s, k=3)

    return spl


def smooth_spline(doy_full, y_full, cfg):
    x = np.asarray(doy_full, float)
    y = np.asarray(y_full, float)

    spl = robust_spline_fit(
        x,
        y,
        s=cfg.get("s", None),
        n_iter=cfg.get("robust_iter", 3),
        zcut=cfg.get("zcut", 3.0),
    )

    if spl is None:
        return None

    yhat = spl(x)

    if cfg.get("clip_nonnegative", True):
        yhat = np.clip(yhat, 0, None)

    return yhat


def ensemble_mean(curves: dict):
    arrs = []

    for v in curves.values():
        if v is None:
            continue

        v = np.asarray(v, float)

        if v.ndim != 1:
            continue

        if len(v) == 0:
            continue

        if not np.all(np.isfinite(v)):
            continue

        arrs.append(v)

    if len(arrs) == 0:
        return None

    return np.clip(np.nanmean(np.vstack(arrs), axis=0), 0, None)


# ============================================================
# 6) Phenology functions
# ============================================================

def _segments_from_mask(mask):
    mask = np.asarray(mask, bool)

    if mask.sum() == 0:
        return []

    diff = np.diff(mask.astype(int))
    starts = np.where(diff == 1)[0] + 1
    ends = np.where(diff == -1)[0]

    if mask[0]:
        starts = np.r_[0, starts]

    if mask[-1]:
        ends = np.r_[ends, len(mask) - 1]

    return [(int(s), int(e)) for s, e in zip(starts, ends)]


def _apply_min_consecutive(segments, min_len):
    if min_len <= 1:
        return segments

    return [(s, e) for s, e in segments if (e - s + 1) >= min_len]


def _pick_segment(segs, use_longest=True):
    if len(segs) == 0:
        return None

    if use_longest:
        lens = np.array([e - s + 1 for s, e in segs], int)
        return segs[int(np.argmax(lens))]

    return segs[0][0], segs[-1][1]


def dorm_min_from_mode(y, cfg, tair=None):
    y = np.asarray(y, float)
    m = np.isfinite(y)

    if m.sum() < 5:
        return np.nan

    mode = cfg.get("gppmin_mode", "quantile")

    if mode == "annual_min":
        return float(np.nanmin(y[m]))

    if mode == "tair_lt0_mean":
        if tair is not None:
            tair = np.asarray(tair, float)
            mm = m & np.isfinite(tair) & (tair < float(cfg.get("tair_for_gppmin", 0.0)))

            if mm.sum() >= 5:
                return float(np.nanmean(y[mm]))

        q = float(cfg.get("gppmin_fallback_quantile", cfg.get("dormant_q", 0.05)))
        return float(np.nanquantile(y[m], q))

    q = float(cfg.get("dormant_q", 0.05))
    q = min(max(q, 0.0), 1.0)

    return float(np.nanquantile(y[m], q))


def curve_stats_for_amp(doy, y, cfg, tair_full=None, stat_mask=None):
    doy = np.asarray(doy, int)
    y = np.asarray(y, float)

    if stat_mask is None:
        stat_mask = np.ones(len(y), dtype=bool)

    stat_mask = np.asarray(stat_mask, bool) & np.isfinite(y)

    if stat_mask.sum() < 10:
        stat_mask = np.isfinite(y)

    a, b = cfg.get("summer_window", (180, 260))
    sm = (doy >= int(a)) & (doy <= int(b)) & stat_mask

    if sm.sum() == 0:
        summer_max = float(np.nanmax(y[stat_mask]))
        doy_gppmax = int(doy[stat_mask][np.nanargmax(y[stat_mask])])
    else:
        summer_max = float(np.nanmax(y[sm]))
        doy_gppmax = int(doy[sm][np.nanargmax(y[sm])])

    if tair_full is not None:
        dorm_min = dorm_min_from_mode(y[stat_mask], cfg, tair=tair_full[stat_mask])
    else:
        dorm_min = dorm_min_from_mode(y[stat_mask], cfg, tair=None)

    amp = float(summer_max - dorm_min) if np.isfinite(summer_max) and np.isfinite(dorm_min) else np.nan

    return dorm_min, summer_max, amp, doy_gppmax


def compute_cup_for_frac(doy, y, thr, mask_start, mask_end, cfg):
    doy = np.asarray(doy, int)
    y = np.asarray(y, float)

    mask_start = np.asarray(mask_start, bool)
    mask_end = np.asarray(mask_end, bool)

    good = np.isfinite(y)
    above = good & (y > float(thr))

    segs = _segments_from_mask(above)
    segs = _apply_min_consecutive(segs, int(cfg.get("min_consecutive_days", 1)))

    seg = _pick_segment(segs, use_longest=bool(cfg.get("use_longest_segment", True)))

    if seg is None:
        return np.nan, np.nan, np.nan

    s0, e0 = seg
    seg_idx = np.arange(s0, e0 + 1)

    ok_start = above[seg_idx] & mask_start[seg_idx]
    ok_end = above[seg_idx] & mask_end[seg_idx]

    if ok_start.sum() == 0 or ok_end.sum() == 0:
        return np.nan, np.nan, np.nan

    s_rel = np.argmax(ok_start)
    e_rel = np.where(ok_end)[0][-1]

    s = int(seg_idx[s_rel])
    e = int(seg_idx[e_rel])

    if e < s:
        return np.nan, np.nan, np.nan

    cup_start = float(doy[s])
    cup_end = float(doy[e])
    cup = float(cup_end - cup_start + 1)

    return cup_start, cup_end, cup


def pick_threshold_dates_from_curve_multi_frac(
    doy_full,
    y_curve,
    cfg,
    tair_mask_start=None,
    tair_mask_end=None,
    tair_gate_mode="none",
    require_crossings_in_warm=True,
    fixed_stats=None,
    plot_id=None,
    curve_key=None,
    tair_full=None,
):
    doy = np.asarray(doy_full, int)

    if y_curve is None:
        return None

    y = np.asarray(y_curve, float)

    if len(y) < 10 or not np.all(np.isfinite(y)):
        return None

    n = len(y)

    if tair_mask_start is None:
        tair_mask_start = np.ones(n, dtype=bool)

    if tair_mask_end is None:
        tair_mask_end = np.ones(n, dtype=bool)

    tair_mask_start = np.asarray(tair_mask_start, bool)
    tair_mask_end = np.asarray(tair_mask_end, bool)

    if not require_crossings_in_warm or tair_gate_mode == "none":
        cross_mask_start = np.ones(n, dtype=bool)
        cross_mask_end = np.ones(n, dtype=bool)
    else:
        cross_mask_start = tair_mask_start
        cross_mask_end = tair_mask_end

    frac_list = [float(f) for f in cfg.get("amp_fracs", [0.10, 0.20, 0.30, 0.25])]
    pcts = [int(round(f * 100)) for f in frac_list]

    # --------------------------
    # Multiyear fixed threshold
    # --------------------------
    if cfg.get("threshold_scope", "annual") == "multiyear" and fixed_stats is not None:
        st = fixed_stats.get((str(plot_id), str(curve_key)), None)

        if st is not None:
            dorm_min = float(st.get("dorm_min", np.nan))
            summer_max = float(st.get("summer_max", np.nan))
            amp = float(st.get("amp", np.nan))

            out = {
                "gpp_max": float(np.nanmax(y)),
                "doy_gppmax": float(doy[int(np.nanargmax(y))]),
                "dorm_min": dorm_min,
                "summer_max": summer_max,
                "amp": amp,
            }

            for pcent in pcts:
                thr = float(st.get(f"thr_{pcent}", np.nan))
                cs, ce, cup = compute_cup_for_frac(doy, y, thr, cross_mask_start, cross_mask_end, cfg)

                out[f"thr_{pcent}"] = thr
                out[f"cup_start_{pcent}"] = cs
                out[f"cup_end_{pcent}"] = ce
                out[f"cup_{pcent}"] = cup

            out["thr_1030"] = float(st.get("thr_1030", np.nan))

            cs_1030 = np.nanmean([
                out.get("cup_start_10", np.nan),
                out.get("cup_start_20", np.nan),
                out.get("cup_start_30", np.nan),
            ])
            ce_1030 = np.nanmean([
                out.get("cup_end_10", np.nan),
                out.get("cup_end_20", np.nan),
                out.get("cup_end_30", np.nan),
            ])

            out["cup_start_1030"] = float(cs_1030) if np.isfinite(cs_1030) else np.nan
            out["cup_end_1030"] = float(ce_1030) if np.isfinite(ce_1030) else np.nan

            if np.isfinite(out["cup_start_1030"]) and np.isfinite(out["cup_end_1030"]):
                out["cup_1030"] = float(out["cup_end_1030"] - out["cup_start_1030"] + 1)
            else:
                out["cup_1030"] = np.nan

            return out

    # --------------------------
    # Annual threshold
    # --------------------------
    if tair_gate_mode == "gate_curve_stats" and require_crossings_in_warm:
        stat_mask = tair_mask_start
    else:
        stat_mask = np.ones(n, dtype=bool)

    dorm_min, summer_max, amp, doy_gppmax = curve_stats_for_amp(
        doy=doy,
        y=y,
        cfg=cfg,
        tair_full=tair_full,
        stat_mask=stat_mask,
    )

    out = {
        "gpp_max": float(summer_max) if np.isfinite(summer_max) else float(np.nanmax(y)),
        "doy_gppmax": float(doy_gppmax),
        "dorm_min": float(dorm_min),
        "summer_max": float(summer_max),
        "amp": float(amp),
    }

    if not np.isfinite(amp) or amp <= 0:
        for pcent in pcts:
            out[f"thr_{pcent}"] = np.nan
            out[f"cup_start_{pcent}"] = np.nan
            out[f"cup_end_{pcent}"] = np.nan
            out[f"cup_{pcent}"] = np.nan

        out["thr_1030"] = np.nan
        out["cup_start_1030"] = np.nan
        out["cup_end_1030"] = np.nan
        out["cup_1030"] = np.nan

        return out

    for frac, pcent in zip(frac_list, pcts):
        thr = float(dorm_min + frac * (summer_max - dorm_min))
        cs, ce, cup = compute_cup_for_frac(doy, y, thr, cross_mask_start, cross_mask_end, cfg)

        out[f"thr_{pcent}"] = thr
        out[f"cup_start_{pcent}"] = cs
        out[f"cup_end_{pcent}"] = ce
        out[f"cup_{pcent}"] = cup

    cs_1030 = np.nanmean([
        out.get("cup_start_10", np.nan),
        out.get("cup_start_20", np.nan),
        out.get("cup_start_30", np.nan),
    ])

    ce_1030 = np.nanmean([
        out.get("cup_end_10", np.nan),
        out.get("cup_end_20", np.nan),
        out.get("cup_end_30", np.nan),
    ])

    out["cup_start_1030"] = float(cs_1030) if np.isfinite(cs_1030) else np.nan
    out["cup_end_1030"] = float(ce_1030) if np.isfinite(ce_1030) else np.nan

    if np.isfinite(out["cup_start_1030"]) and np.isfinite(out["cup_end_1030"]):
        out["cup_1030"] = float(out["cup_end_1030"] - out["cup_start_1030"] + 1)
    else:
        out["cup_1030"] = np.nan

    thr_1030 = np.nanmean([
        out.get("thr_10", np.nan),
        out.get("thr_20", np.nan),
        out.get("thr_30", np.nan),
    ])

    out["thr_1030"] = float(thr_1030) if np.isfinite(thr_1030) else np.nan

    return out


# ============================================================
# 7) Multiyear fixed thresholds
# ============================================================

def select_year_pool(cfg, years):
    years = list(map(int, years))
    pret = set(map(int, cfg.get("pretreat_years", [])))
    pool = cfg.get("threshold_year_pool", "pretreatment")

    if pool == "pretreatment":
        return [y for y in years if y in pret]

    if pool == "treatment":
        return [y for y in years if y not in pret]

    return years


def build_multiyear_thresholds(curve_store, cfg):
    years_pool = set(select_year_pool(cfg, cfg["years"]))
    curve_keys = cfg.get("curve_keys", [])

    frac_list = [float(f) for f in cfg.get("amp_fracs", [0.10, 0.20, 0.30, 0.25])]
    pcts = [int(round(f * 100)) for f in frac_list]

    fixed_stats = {}

    for plot in cfg["plots"]:
        yrs = [
            int(y)
            for y in cfg["years"]
            if (str(plot), int(y)) in curve_store and int(y) in years_pool
        ]

        if len(yrs) == 0:
            continue

        for ck in curve_keys:
            dorms = []
            summers = []

            for y in yrs:
                rec = curve_store[(str(plot), int(y))]
                doy_full = rec["doy_full"]
                tair_full = rec.get("tair_full", None)
                curve = rec["curves"].get(ck, None)

                if curve is None:
                    continue

                curve = np.asarray(curve, float)

                if not np.all(np.isfinite(curve)):
                    continue

                dorm_min, summer_max, amp, _ = curve_stats_for_amp(
                    doy=doy_full,
                    y=curve,
                    cfg=cfg,
                    tair_full=tair_full,
                    stat_mask=np.ones(len(doy_full), dtype=bool),
                )

                if np.isfinite(dorm_min) and np.isfinite(summer_max):
                    dorms.append(float(dorm_min))
                    summers.append(float(summer_max))

            if len(dorms) == 0 or len(summers) == 0:
                continue

            dorm_fix = float(np.mean(dorms))
            summer_fix = float(np.mean(summers))
            amp_fix = float(summer_fix - dorm_fix)

            out = {
                "dorm_min": dorm_fix,
                "summer_max": summer_fix,
                "amp": amp_fix,
            }

            for frac, pcent in zip(frac_list, pcts):
                out[f"thr_{pcent}"] = (
                    float(dorm_fix + frac * (summer_fix - dorm_fix))
                    if np.isfinite(amp_fix)
                    else np.nan
                )

            out["thr_1030"] = float(np.mean([
                out.get("thr_10", np.nan),
                out.get("thr_20", np.nan),
                out.get("thr_30", np.nan),
            ]))

            fixed_stats[(str(plot), str(ck))] = out

    return fixed_stats


# ============================================================
# 8) Pretreatment anomaly
# ============================================================

def add_pretreatment_anomaly(df_out, cfg):
    pretreat_mode = cfg.get("pretreat_mode", "years_mean")
    pretreat_years = cfg.get("pretreat_years", [])
    grouping = cfg.get("pretreat_grouping", "method_plot")

    if pretreat_mode not in ["years_mean", "single_year"]:
        raise ValueError("pretreat_mode must be 'years_mean' or 'single_year'.")

    if pretreat_mode == "single_year":
        if len(pretreat_years) != 1:
            raise ValueError("pretreat_mode='single_year' requires one year.")
    else:
        if len(pretreat_years) == 0:
            raise ValueError("pretreat_mode='years_mean' requires pretreat_years.")

    pretreat_years_set = set(map(int, pretreat_years))

    value_cols = []
    for c in df_out.columns:
        if not pd.api.types.is_numeric_dtype(df_out[c]):
            continue

        if c.startswith((
            "temp_",
            "value_sum",
            "gpp_sum",
            "gcc_sum",
            "gpp_max_",
            "doy_gppmax_",
            "cup_start_",
            "cup_end_",
            "cup_",
            "thr_",
            "dorm_min_",
            "amp_",
            "alpha_",
            "summer_max_",
        )):
            value_cols.append(c)

    if len(value_cols) == 0:
        out = df_out.copy()
        out["pretreat_mode"] = pretreat_mode
        out["pretreat_years"] = ",".join(map(str, sorted(list(pretreat_years_set))))
        out["pretreat_grouping"] = grouping
        return out

    base = df_out[df_out["year"].isin(pretreat_years_set)].copy()

    if base.empty:
        out = df_out.copy()
        out["pretreat_mode"] = pretreat_mode
        out["pretreat_years"] = ",".join(map(str, sorted(list(pretreat_years_set))))
        out["pretreat_grouping"] = grouping
        return out

    def _available(cols):
        return [c for c in cols if c in df_out.columns]

    if grouping == "method_plot":
        group_cols = _available(["method", "plot"])
    elif grouping == "method":
        group_cols = _available(["method"])
    else:
        raise ValueError("pretreat_grouping must be 'method_plot' or 'method'.")

    if len(group_cols) == 0:
        group_cols = ["plot"] if "plot" in df_out.columns else []

    if len(group_cols) == 0:
        base_mean = base[value_cols].mean(numeric_only=True).to_frame().T
        base_mean = base_mean.rename(columns={c: f"{c}_base" for c in value_cols})

        out = df_out.copy()

        for c in value_cols:
            out[f"{c}_base"] = float(base_mean[f"{c}_base"].iloc[0])
            out[f"{c}_anom"] = out[c] - out[f"{c}_base"]

    else:
        base_tbl = (
            base.groupby(group_cols, as_index=False)[value_cols]
            .mean()
            .rename(columns={c: f"{c}_base" for c in value_cols})
        )

        out = df_out.merge(base_tbl, on=group_cols, how="left")

        for c in value_cols:
            out[f"{c}_anom"] = out[c] - out[f"{c}_base"]

    out["pretreat_mode"] = pretreat_mode
    out["pretreat_years"] = ",".join(map(str, sorted(list(pretreat_years_set))))
    out["pretreat_grouping"] = grouping

    return out


# ============================================================
# 9) Plotting panels
# ============================================================

def plot_panels(panel_records, cfg, sps=""):
    plots = cfg["plots"]
    years = cfg["years"]

    nrow = len(plots)
    ncol = len(years)

    fig = plt.figure(figsize=(cfg["fig_cell_w"] * ncol, cfg["fig_cell_h"] * nrow))
    axes = fig.subplots(nrow, ncol, sharex=True)

    if nrow == 1 and ncol == 1:
        axes = np.array([[axes]])
    elif nrow == 1:
        axes = axes.reshape(1, -1)
    elif ncol == 1:
        axes = axes.reshape(-1, 1)

    rec_map = {(r["plot"], r["year"]): r for r in panel_records}

    thrS = float(cfg.get("plot_tair_threshold_start", 0.0))
    thrE = float(cfg.get("plot_tair_threshold_end", 0.0))
    tag = f"S{thrS:g}_E{thrE:g}"

    curve_keys = cfg.get("curve_keys", [])

    for i, p in enumerate(plots):
        for j, y in enumerate(years):
            ax = axes[i, j]
            r = rec_map.get((str(p), int(y)), None)

            if r is None:
                ax.axis("off")
                continue

            ax.scatter(
                r["doy_raw"],
                r["value_raw"],
                s=cfg["raw_point_size"],
                alpha=cfg["raw_alpha"],
                label="Raw",
            )

            line_objs = {}

            for ck in curve_keys:
                arr = r["curves"].get(ck, None)

                if arr is None:
                    continue

                lw = 1.0

                if ck.startswith("final_"):
                    lw = 1.6

                if ck == "weibull":
                    lw = 1.2

                line_objs[ck] = ax.plot(r["doy_full"], arr, lw=lw, label=ck)[0]

            if cfg.get("plot_cup_lines", True):
                for ck in curve_keys:
                    if ck not in line_objs:
                        continue

                    col = line_objs[ck].get_color()

                    cs25 = r.get(f"cup_start_{ck}_25_{tag}", np.nan)
                    ce25 = r.get(f"cup_end_{ck}_25_{tag}", np.nan)
                    cs1030 = r.get(f"cup_start_{ck}_1030_{tag}", np.nan)
                    ce1030 = r.get(f"cup_end_{ck}_1030_{tag}", np.nan)

                    if np.isfinite(cs25):
                        ax.axvline(cs25, ls=":", lw=1.0, color=col, alpha=0.85)

                    if np.isfinite(ce25):
                        ax.axvline(ce25, ls=":", lw=1.0, color=col, alpha=0.85)

                    if np.isfinite(cs1030):
                        ax.axvline(cs1030, ls="--", lw=1.0, color=col, alpha=0.65)

                    if np.isfinite(ce1030):
                        ax.axvline(ce1030, ls="--", lw=1.0, color=col, alpha=0.65)

            if cfg.get("plot_tair0_vlines", True):
                spring0, fall0 = tair0_vline_doys(r.get("tair_full", None), r["doy_full"])

                if np.isfinite(spring0):
                    ax.axvline(spring0, color="k", lw=0.8, ls="-", alpha=0.25)

                if np.isfinite(fall0):
                    ax.axvline(fall0, color="k", lw=0.8, ls="-", alpha=0.25)

            if i == 0:
                ax.set_title(str(y), fontsize=10)

            if j == 0:
                ax.set_ylabel(p, fontsize=10)

            ax.tick_params(labelsize=8)

    fig.supxlabel("Day of Year", fontsize=12)
    fig.supylabel(cfg.get("y_label", "Value"), fontsize=12)

    handles, labels = [], []

    for ax in axes.flatten():
        h, l = ax.get_legend_handles_labels()

        if len(h) > 0:
            handles, labels = h, l
            break

    if len(handles) > 0:
        fig.legend(
            handles,
            labels,
            loc="upper center",
            ncol=min(6, len(labels)),
            frameon=False,
            fontsize=9,
        )

    fig.tight_layout(rect=[0, 0, 1, 0.965])

    out_png = os.path.join(cfg["out_dir"], cfg["out_png"].format(sps=sps))
    fig.savefig(out_png, dpi=cfg["dpi"])
    plt.close(fig)

    print("Saved figure:", out_png)


# ============================================================
# 10) Main function
# ============================================================

def run_phenology_pipeline(cfg, sps):
    ensure_dir(cfg["out_dir"])

    df_raw = pd.read_excel(cfg["input_xlsx"].format(sps=sps), sheet_name=cfg["sheet_name"])
    df = detect_and_normalize(df_raw, cfg)

    plots = [str(p) for p in cfg["plots"]]
    years = [int(y) for y in cfg["years"]]

    map_co2 = {p: cfg["co2_by_plot"][i] for i, p in enumerate(plots)}
    map_warm = {p: cfg["warming_by_plot"][i] for i, p in enumerate(plots)}

    forcing_daily = {}

    for p in plots:
        try:
            forcing_daily[p] = load_forcing_daily(p, cfg)
        except Exception as e:
            print(f"[forcing] WARNING: failed to load forcing for {p}: {e}")
            forcing_daily[p] = None

    start_opts = [float(x) for x in cfg.get("tair_threshold_options_start", [0.0])]
    end_opts = [float(x) for x in cfg.get("tair_threshold_options_end", [0.0])]

    if cfg.get("run_all_tair_threshold_pairs", False):
        tair_pairs = [(a, b) for a in start_opts for b in end_opts]
    else:
        tair_pairs = [(
            float(cfg.get("plot_tair_threshold_start", start_opts[0])),
            float(cfg.get("plot_tair_threshold_end", end_opts[0])),
        )]

    def _alpha(value_sum, gmax, cup):
        return alpha_from(
            value_sum,
            gmax,
            cup,
            calc_alpha=cfg.get("calc_alpha", True),
        )

    frac_list = [float(f) for f in cfg.get("amp_fracs", [0.10, 0.20, 0.30, 0.25])]
    frac_pcts = [int(round(f * 100)) for f in frac_list]
    summer_window_str = f"{cfg['summer_window'][0]}-{cfg['summer_window'][1]}"

    curve_store = {}
    panel_records = []

    # --------------------------
    # Pass 1: build curves
    # --------------------------
    for p in plots:
        for y in years:
            sub = df[(df["plot"] == p) & (df["year"] == y)].copy()

            if sub.empty:
                continue

            sub = sub.sort_values("doy")

            doy_raw = sub["doy"].to_numpy()
            value_raw = sub["value"].to_numpy()
            value_sum = float(np.nansum(value_raw))

            doy_full, y_full, _ = to_full_daily_series(doy_raw, value_raw, y)

            tair_full = tair_full_for_year(forcing_daily.get(p, None), y, doy_full)

            weibull_fit, _, _, _ = fit_year_weibull_single_or_double(
                doy_full,
                y_full,
                cfg,
                sps,
                plot=p,
                year=y,
            )

            hants = smooth_hants(doy_full, y_full, cfg["hants"])
            sg = smooth_savgol(y_full, cfg["savgol"])
            sp = smooth_spline(doy_full, y_full, cfg["spline"])

            final_hss = ensemble_mean({
                "hants": hants,
                "savgol": sg,
                "spline": sp,
            })

            final_whs = ensemble_mean({
                "weibull": weibull_fit,
                "hants": hants,
                "savgol": sg,
            })

            curves = {
                "weibull": weibull_fit,
                "hants": hants,
                "savgol": sg,
                "spline": sp,
                "final_hss": final_hss,
                "final_whs": final_whs,
            }

            curve_store[(p, y)] = {
                "plot": p,
                "year": y,
                "doy_raw": doy_raw,
                "value_raw": value_raw,
                "value_sum": value_sum,
                "doy_full": doy_full,
                "tair_full": tair_full,
                "curves": curves,
            }

            panel_records.append({
                "plot": p,
                "year": y,
                "doy_raw": doy_raw,
                "value_raw": value_raw,
                "doy_full": doy_full,
                "tair_full": tair_full,
                "curves": curves,
            })

    fixed_stats = None

    if cfg.get("threshold_scope", "annual") == "multiyear":
        fixed_stats = build_multiyear_thresholds(curve_store, cfg)

        print(
            f"[threshold_scope=multiyear] built fixed thresholds for "
            f"{len(fixed_stats)} (plot,curve) pairs using "
            f"year pool='{cfg.get('threshold_year_pool', 'pretreatment')}'."
        )

    # --------------------------
    # Pass 2: phenology metrics
    # --------------------------
    out_rows = []

    gate_mode = cfg.get("tair_gate_mode", "none")
    require_warm = bool(cfg.get("require_crossings_in_warm", False))

    value_sum_col = cfg.get("value_sum_col", "value_sum")

    for (p, y), rec in curve_store.items():
        value_sum = rec["value_sum"]
        doy_full = rec["doy_full"]
        tair_full = rec["tair_full"]

        if forcing_daily.get(p, None) is not None:
            temp_mean, temp_gs_mean, temp_max = summarize_temp(
                forcing_daily[p],
                y,
                cfg["gs_doy_min"],
                cfg["gs_doy_max"],
            )
        else:
            temp_mean, temp_gs_mean, temp_max = np.nan, np.nan, np.nan

        for tair_thr_start, tair_thr_end in tair_pairs:
            tmask_start = tair_mask_for_year(
                forcing_daily.get(p, None),
                y,
                doy_full,
                tair_thr_start,
            )

            tmask_end = tair_mask_for_year(
                forcing_daily.get(p, None),
                y,
                doy_full,
                tair_thr_end,
            )

            row = {
                "plot": p,
                "year": int(y),
                "co2": map_co2.get(p, np.nan),
                "warming": map_warm.get(p, np.nan),

                "tair_threshold_start": float(tair_thr_start),
                "tair_threshold_end": float(tair_thr_end),
                "tair_gate_mode": gate_mode,
                "require_crossings_in_warm": require_warm,

                "threshold_scope": cfg.get("threshold_scope", "annual"),
                "threshold_year_pool": cfg.get("threshold_year_pool", "pretreatment"),
                "gppmin_mode": cfg.get("gppmin_mode", "quantile"),

                "dormant_q": float(cfg.get("dormant_q", 0.05)),
                "summer_window": summer_window_str,
                "min_consecutive_days": int(cfg.get("min_consecutive_days", 1)),
                "use_longest_segment": bool(cfg.get("use_longest_segment", True)),

                "temp_mean": temp_mean,
                "temp_gs_mean": temp_gs_mean,
                "temp_max": temp_max,

                "value_sum": float(value_sum),
                value_sum_col: float(value_sum),

                "hants_K": cfg["hants"]["K"],
                "hants_iter": cfg["hants"]["robust_iter"],
                "savgol_window": cfg["savgol"]["window"],
                "savgol_polyorder": cfg["savgol"]["polyorder"],
                "spline_iter": cfg["spline"]["robust_iter"],

                "method": "6curves",
            }

            row["gpp_max_hssM"] = np.nan
            row["doy_gppmax_hssM"] = np.nan
            row["gpp_max_whsM"] = np.nan
            row["doy_gppmax_whsM"] = np.nan

            for suf in [str(pct) for pct in frac_pcts] + ["1030"]:
                for tagM in ["hssM", "whsM"]:
                    row[f"cup_start_{tagM}_{suf}"] = np.nan
                    row[f"cup_end_{tagM}_{suf}"] = np.nan
                    row[f"cup_{tagM}_{suf}"] = np.nan
                    row[f"alpha_{tagM}_{suf}"] = np.nan

            for ck in cfg["curve_keys"]:
                ph = pick_threshold_dates_from_curve_multi_frac(
                    doy_full=doy_full,
                    y_curve=rec["curves"].get(ck, None),
                    cfg=cfg,
                    tair_mask_start=tmask_start,
                    tair_mask_end=tmask_end,
                    tair_gate_mode=gate_mode,
                    require_crossings_in_warm=require_warm,
                    fixed_stats=fixed_stats,
                    plot_id=p,
                    curve_key=ck,
                    tair_full=tair_full,
                )

                row[f"gpp_max_{ck}"] = np.nan
                row[f"doy_gppmax_{ck}"] = np.nan
                row[f"dorm_min_{ck}"] = np.nan
                row[f"summer_max_{ck}"] = np.nan
                row[f"amp_{ck}"] = np.nan

                for pcent in frac_pcts:
                    row[f"thr_{ck}_{pcent}"] = np.nan
                    row[f"cup_start_{ck}_{pcent}"] = np.nan
                    row[f"cup_end_{ck}_{pcent}"] = np.nan
                    row[f"cup_{ck}_{pcent}"] = np.nan
                    row[f"alpha_{ck}_{pcent}"] = np.nan

                row[f"thr_{ck}_1030"] = np.nan
                row[f"cup_start_{ck}_1030"] = np.nan
                row[f"cup_end_{ck}_1030"] = np.nan
                row[f"cup_{ck}_1030"] = np.nan
                row[f"alpha_{ck}_1030"] = np.nan

                if ph is None:
                    continue

                row[f"gpp_max_{ck}"] = ph.get("gpp_max", np.nan)
                row[f"doy_gppmax_{ck}"] = ph.get("doy_gppmax", np.nan)
                row[f"dorm_min_{ck}"] = ph.get("dorm_min", np.nan)
                row[f"summer_max_{ck}"] = ph.get("summer_max", np.nan)
                row[f"amp_{ck}"] = ph.get("amp", np.nan)

                for pcent in frac_pcts:
                    row[f"thr_{ck}_{pcent}"] = ph.get(f"thr_{pcent}", np.nan)
                    row[f"cup_start_{ck}_{pcent}"] = ph.get(f"cup_start_{pcent}", np.nan)
                    row[f"cup_end_{ck}_{pcent}"] = ph.get(f"cup_end_{pcent}", np.nan)
                    row[f"cup_{ck}_{pcent}"] = ph.get(f"cup_{pcent}", np.nan)
                    row[f"alpha_{ck}_{pcent}"] = _alpha(
                        value_sum,
                        row[f"gpp_max_{ck}"],
                        row[f"cup_{ck}_{pcent}"],
                    )

                row[f"thr_{ck}_1030"] = ph.get("thr_1030", np.nan)
                row[f"cup_start_{ck}_1030"] = ph.get("cup_start_1030", np.nan)
                row[f"cup_end_{ck}_1030"] = ph.get("cup_end_1030", np.nan)
                row[f"cup_{ck}_1030"] = ph.get("cup_1030", np.nan)
                row[f"alpha_{ck}_1030"] = _alpha(
                    value_sum,
                    row[f"gpp_max_{ck}"],
                    row[f"cup_{ck}_1030"],
                )

                for pr in panel_records:
                    if pr["plot"] == p and pr["year"] == y:
                        tag = f"S{tair_thr_start:g}_E{tair_thr_end:g}"

                        pr[f"cup_start_{ck}_25_{tag}"] = ph.get("cup_start_25", np.nan)
                        pr[f"cup_end_{ck}_25_{tag}"] = ph.get("cup_end_25", np.nan)
                        pr[f"cup_start_{ck}_1030_{tag}"] = ph.get("cup_start_1030", np.nan)
                        pr[f"cup_end_{ck}_1030_{tag}"] = ph.get("cup_end_1030", np.nan)

                        break

            HSS_members = ["hants", "savgol", "spline"]
            WHS_members = ["weibull", "hants", "savgol"]

            gmax_hssM, doy_hssM = ensemble_gppmax_from_metrics(row, HSS_members)
            gmax_whsM, doy_whsM = ensemble_gppmax_from_metrics(row, WHS_members)

            row["gpp_max_hssM"] = gmax_hssM
            row["doy_gppmax_hssM"] = doy_hssM
            row["gpp_max_whsM"] = gmax_whsM
            row["doy_gppmax_whsM"] = doy_whsM

            suffix_list = [str(pct) for pct in frac_pcts] + ["1030"]

            for suf in suffix_list:
                cs, ce, cup = ensemble_cup_from_metrics(row, HSS_members, suf)
                row[f"cup_start_hssM_{suf}"] = cs
                row[f"cup_end_hssM_{suf}"] = ce
                row[f"cup_hssM_{suf}"] = cup
                row[f"alpha_hssM_{suf}"] = alpha_from(
                    value_sum,
                    row["gpp_max_hssM"],
                    cup,
                    calc_alpha=cfg.get("calc_alpha", True),
                )

                cs, ce, cup = ensemble_cup_from_metrics(row, WHS_members, suf)
                row[f"cup_start_whsM_{suf}"] = cs
                row[f"cup_end_whsM_{suf}"] = ce
                row[f"cup_whsM_{suf}"] = cup
                row[f"alpha_whsM_{suf}"] = alpha_from(
                    value_sum,
                    row["gpp_max_whsM"],
                    cup,
                    calc_alpha=cfg.get("calc_alpha", True),
                )

            out_rows.append(row)

    df_out = (
        pd.DataFrame(out_rows)
        .sort_values(["plot", "year", "tair_threshold_start", "tair_threshold_end"])
        .reset_index(drop=True)
    )

    if bool(cfg.get("pretreat_enable", True)):
        df_out = add_pretreatment_anomaly(df_out, cfg)

    out_xlsx = os.path.join(cfg["out_dir"], cfg["out_xlsx"].format(sps=sps))
    df_out.to_excel(out_xlsx, index=False)

    print("Saved table:", out_xlsx)

    plot_panels(panel_records, cfg, sps)

    return df_out

In [3]:
# ============================================================
# Run GPP
# ============================================================

GPP_CONFIG = COMMON_CONFIG.copy()

GPP_CONFIG.update({
    "input_xlsx": "../../data_results/2_results/2-1_simu_gpp_time_series/TECO-SPRUCE_DA_{sps}_2011-2021.xlsx",
    "sps_list": ["ecosystem", "tree", "shrub", "sphagnum"],
    "years": list(range(2011, 2022)),

    "out_dir": "../../data_results/2_results/2-3_gppmax_cup_6methods",
    "out_xlsx": "calculated_GPPmax_CUP_ampFracs_10_20_30_25_thresholdScope_{sps}.xlsx",
    "out_png": "timeseries_panels_{sps}.png",

    "y_label": "GPP (g C m$^{-2}$ d$^{-1}$)",
    "value_sum_col": "gpp_sum",
    "calc_alpha": True,

    "pretreat_enable": True,
    "pretreat_years": [2011, 2012, 2013],
    "pretreat_grouping": "method_plot",
})

for sps in GPP_CONFIG["sps_list"]:
    print("\n==============================")
    print("Running GPP:", sps)
    df_gpp = run_phenology_pipeline(GPP_CONFIG, sps)


Running GPP: ecosystem
Saved table: ../../data_results/2_results/2-3_gppmax_cup_6methods/calculated_GPPmax_CUP_ampFracs_10_20_30_25_thresholdScope_ecosystem.xlsx
Saved figure: ../../data_results/2_results/2-3_gppmax_cup_6methods/timeseries_panels_ecosystem.png

Running GPP: tree
Saved table: ../../data_results/2_results/2-3_gppmax_cup_6methods/calculated_GPPmax_CUP_ampFracs_10_20_30_25_thresholdScope_tree.xlsx
Saved figure: ../../data_results/2_results/2-3_gppmax_cup_6methods/timeseries_panels_tree.png

Running GPP: shrub
Saved table: ../../data_results/2_results/2-3_gppmax_cup_6methods/calculated_GPPmax_CUP_ampFracs_10_20_30_25_thresholdScope_shrub.xlsx
Saved figure: ../../data_results/2_results/2-3_gppmax_cup_6methods/timeseries_panels_shrub.png

Running GPP: sphagnum
Saved table: ../../data_results/2_results/2-3_gppmax_cup_6methods/calculated_GPPmax_CUP_ampFracs_10_20_30_25_thresholdScope_sphagnum.xlsx
Saved figure: ../../data_results/2_results/2-3_gppmax_cup_6methods/timeseries_pa